In [12]:
import pandas as pd
from rapidfuzz import process, fuzz
from clean_data import apply_abbreviation, clean_results_df, clean_stats_df, insert_surface
import numpy as np

In [ ]:
#### Add Surface Type To Tournaments

In [5]:
tournament_surfaces = {
    'Brisbane International presented by Evie': 'Hard',
    'Bank of China Hong Kong Tennis Open': 'Hard',
    'Adelaide International': 'Hard',
    'ASB Classic': 'Hard',
    'Australian Open': 'Hard',
    'Open Occitanie': 'Hard (Indoor)',
    'Dallas Open': 'Hard (Indoor)',
    'ABN AMRO Open': 'Hard (Indoor)',
    'Open 13 Provence': 'Hard (Indoor)',
    'Delray Beach Open': 'Hard',
    'IEB+ Argentina Open': 'Clay',
    'Qatar ExxonMobil Open': 'Hard',
    'Rio Open presented by Claro': 'Clay',
    'Dubai Duty Free Tennis Championships': 'Hard',
    'Abierto Mexicano Telcel presentado por HSBC': 'Hard',
    'Movistar Chile Open': 'Clay',
    'BNP Paribas Open': 'Hard',
    'Miami Open presented by Itau': 'Hard',
    "Fayez Sarofim & Co. U.S. Men's Clay Court Championship": 'Clay',
    'Grand Prix Hassan II': 'Clay',
    'Tiriac Open presented by UniCredit Bank': 'Clay',
    'Rolex Monte-Carlo Masters': 'Clay',
    'Barcelona Open Banc Sabadell': 'Clay',
    'BMW Open by Bitpanda': 'Clay',
    'Mutua Madrid Open': 'Clay',
    "Internazionali BNL d'Italia": 'Clay',
    'Bitpanda Hamburg Open': 'Clay',
    'Gonet Geneva Open': 'Clay',
    'Roland Garros': 'Clay',
    'BOSS OPEN': 'Grass',
    'Libema Open': 'Grass',
    'HSBC Championships': 'Grass',
    'Terra Wortmann Open': 'Grass',
    'Wimbledon': 'Grass',
    'Hamburg': 'Clay',
    'Newport': 'Grass',
    'Bastad': 'Clay',
    'Gstaad': 'Clay',
    'Umag': 'Clay',
    'Atlanta': 'Hard',
    'Kitzbuhel': 'Clay',
    'Washington': 'Hard',
    'ATP Masters 1000 Canada': 'Hard',
    'ATP Masters 1000 Cincinnati': 'Hard',
    'Winston-Salem': 'Hard',
    'US Open': 'Hard',
    'Chengdu': 'Hard',
    'Hangzhou': 'Hard',
    'Tokyo': 'Hard',
    'Beijing': 'Hard',
    'ATP Masters 1000 Shanghai': 'Hard',
    'Almaty': 'Hard',
    'Antwerp': 'Hard',
    'Stockholm': 'Hard (Indoor)',
    'Vienna': 'Hard (Indoor)',
    'Basel': 'Hard (Indoor)',
    'ATP Masters 1000 Paris': 'Hard (Indoor)',
    'Belgrade': 'Clay',
    'Metz': 'Hard (Indoor)',
    'Nitto ATP Finals': 'Hard (Indoor)',
    'Next Gen ATP Finals': 'Hard (Indoor)'
}

Make more robust by joining players with their IDs from df_top_500_players


The player names in ```df_top_200_players``` is of the form "M. Last Name". We need a way to match the names in ```df_tournament_results```

---
---
---

In [ ]:
# Merging and getting id for player1
merged_tourn_result = pd.merge(df_tournament_results, df_top_500_players[["Name", "id"]], left_on="player_1", right_on="Name", how="left")
merged_tourn_result = merged_tourn_result.rename(columns={"id": "player_1_id"}).drop(columns="Name")

# Merging and getting id for player2
merged_tourn_result = pd.merge(merged_tourn_result, df_top_500_players[["Name", "id"]], left_on="player_2", right_on="Name", how="left")
merged_tourn_result = merged_tourn_result.rename(columns={"id": "player_2_id"}).drop(columns="Name")

In [ ]:
merged_tourn_result.isnull().sum()

---
---
---

## Clean Results

In [7]:
df_results = pd.read_csv('s3://matchedge-pipeline/data/raw/all_results.csv')

In [8]:
df_results_trial = clean_results_df(df_results)

Unmatched: Jacob Bradshaw (Best guess: M. Braswell, Score: 48.0)
Unmatched: Orlando Luz (Best guess: L. Tu, Score: 48.857142857142854)
Unmatched: Rudy Quan (Best guess: F. Sun, Score: 49.090909090909086)
Unmatched: Jiri Vesely (Best guess: J. Fearnley, Score: 45.45454545454546)
Unmatched: Omni Kumar (Best guess: L. Neumayer, Score: 47.61904761904761)
Unmatched: Omni Kumar (Best guess: L. Neumayer, Score: 47.61904761904761)


In [4]:
df_results_trial

,match_date,player_1,player_2,duration,match_round,player_1_scores,player_2_scores,winner,result,match_id,...,p1_set3,p1_set4,p1_set5,p2_set1,p2_set2,p2_set3,p2_set4,p2_set5,best_of,winner_id
0,05-01-2025,J. Lehecka,R. Opelka,00:13:09,Final,4,1,Jiri Lehecka,RET,ms001,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,3,l0bv
1,04-01-2025,J. Lehecka,G. Dimitrov,01:23:33,Semifinals,6 4,4 4,Jiri Lehecka,RET,ms003,...,NaN,NaN,NaN,4,4,NaN,NaN,NaN,3,l0bv
2,04-01-2025,R. Opelka,G. Mpetshi Perricard,01:26:13,Semifinals,6 7,3 6(4),Reilly Opelka,Completed,ms002,...,NaN,NaN,NaN,3,6(4),NaN,NaN,NaN,3,o522
3,03-01-2025,R. Opelka,N. Djokovic,01:40:11,Quarterfinals,7 6,6(6) 3,Reilly Opelka,Completed,ms004,...,NaN,NaN,NaN,6(6),3,NaN,NaN,NaN,3,o522
4,03-01-2025,G. Dimitrov,J. Thompson,00:43:03,Quarterfinals,6 2,1 1,Grigor Dimitrov,RET,ms007,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,3,d875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3997,22-12-2024,N. Basavareddy,J. Shang,01:33,Round Robin -,3(4) 4 4 4,4 2 2 1,Nishesh Basavareddy,Completed,ms020,...,4,4,NaN,4,2,2,1,NaN,5,b0nn
3998,22-12-2024,J. Fonseca,A. Fils,01:50,Round Robin -,3(9) 4 4 1 4,4 2 1 4 1,Joao Fonseca,Completed,ms010,...,4,1,4,4,2,1,4,1,5,f0fv
3999,22-12-2024,L. Tien,J. Mensik,02:17,Round Robin -,4 4 2 2 4,3(6) 3(3) 4 4 3(8),Learner Tien,Completed,ms011,...,2,2,4,3(6),3(3),4,4,3(8),5,t0ha
4000,22-12-2024,A. Michelsen,N. Basavareddy,01:53,Round Robin -,2 4 4 4,4 3(5) 3(4) 2,Alex Michelsen,Completed,ms018,...,4,4,NaN,4,3(5),3(4),2,NaN,5,m0qi


Save cleaned results data

In [9]:
df_results_trial.to_csv('/Users/samueleferrucci/Documents/Coding/Projects/Tennis ML/data/clean/all_results.csv', sep=',', columns=df_results_trial.columns, index=False)
df_results_trial.to_csv('s3://matchedge-pipeline/data/clean/all_results.csv', sep=',', columns=df_results_trial.columns, index=False)

---

## Clean Stats

### Combine GS Stats and Earlier Stats

In [ ]:
df_top = pd.read_csv("s3://matchedge-pipeline/data/clean/top_500_players.csv")
df_stat = pd.read_csv('s3://matchedge-pipeline/data/raw/all_stats.csv')
df_stat_GS = pd.read_csv('s3://matchedge-pipeline/data/raw/GS_stats.csv')

In [5]:
df_stat.dropna(how="all", inplace=True)
df_stat_GS.dropna(how="all", inplace=True)

In [6]:
df_stat.columns == df_stat_GS.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True, False, False])

In [7]:
df_stat.rename(columns={"p1_2ns_serve_avg_speed": "p1_2nd_serve_average_speed", 
                             "p1_1st_serve_avg_speed": "p1_1st_serve_average_speed",
                             "p2_2ns_serve_avg_speed": "p2_2nd_serve_average_speed",
                             "p2_1st_serve_avg_speed": "p2_1st_serve_average_speed"}, 
                    inplace=True)

In [8]:
df_stat.columns == df_stat_GS.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [9]:
df_all_stats = pd.concat([df_stat, df_stat_GS])

Save the combined 'all_stats' with the GS stats

In [ ]:
df_all_stats.to_csv('/Users/samueleferrucci/Documents/Coding/Projects/Tennis ML/data/raw/all_stats_GS.csv', sep=',', columns=df_all_stats.columns, index=False)
df_all_stats.to_csv('s3://matchedge-pipeline/data/raw/all_stats_GS.csv', sep=',', columns=df_all_stats.columns, index=False)

In [2]:
df_all_stats = pd.read_csv('/Users/samueleferrucci/Documents/Coding/Projects/Tennis ML/data/raw/all_stats_GS.csv', sep=',')

### Clean All The Stats

In [3]:
df_all_stats_trial = clean_stats_df(df_all_stats)

In [4]:
df_all_stats_trial.head()

,match_id,tournament_id,player_1,player_2,p1_id,p2_id,p1_serve_rating,p1_aces,p1_double_faults,p1_first_serve,...,p2_service_points_won,p2_return_points_won,p2_total_points_won,p2_max_speed,p2_1st_serve_average_speed,p2_2nd_serve_average_speed,p2_break_point_opportunities,p1_break_point_opportunities,p1_net_points_played,p2_net_points_played
0,ms001,339.0,R. Opelka,J. Lehecka,o522,l0bv,194.0,0.0,0.0,1.00,...,0.80,0.56,0.71,NaN,NaN,NaN,1.0,0.0,0.0,2.0
1,ms003,339.0,J. Lehecka,G. Dimitrov,l0bv,d875,301.0,4.0,3.0,0.50,...,0.73,0.25,0.49,NaN,NaN,NaN,0.0,2.0,5.0,3.0
2,ms002,339.0,R. OpelkaR. OPELKA,G. Mpetshi ...G. MPETSHI PERRICARD,o522,m0gz,328.0,12.0,2.0,0.70,...,0.68,0.20,0.45,NaN,NaN,NaN,4.0,2.0,17.0,17.0
3,ms004,339.0,N. Djokovic,R. Opelka,d643,o522,298.0,8.0,1.0,0.73,...,0.80,0.32,0.54,NaN,NaN,NaN,5.0,1.0,4.0,7.0
4,ms007,339.0,J. Thompson,G. Dimitrov,tc61,d875,187.0,1.0,1.0,0.46,...,0.77,0.50,0.65,NaN,NaN,NaN,4.0,0.0,8.0,5.0


Save to cleaned data

In [6]:
df_all_stats_trial.to_csv('/Users/samueleferrucci/Documents/Coding/Projects/Tennis ML/data/clean/all_stats.csv', sep=',', columns=df_all_stats_trial.columns, index=False)
df_all_stats_trial.to_csv("s3://matchedge-pipeline/data/clean/all_stats.csv", sep=',', columns=df_all_stats_trial.columns, index=False)

---

## Clean All Tournaments

In [10]:
df_all_tournament = pd.read_csv("s3://matchedge-pipeline/data/raw/All_Tournaments.csv", sep=',')

In [11]:
df_all_tournament

,id,name,level,location,surface,end_date,url
0,339,Brisbane International presented by Evie,ATP 250,"Brisbane, Australia",hard,2025-01-05,https://www.atptour.com/en/tournaments/brisban...
1,336,Bank of China Hong Kong Tennis Open,ATP 250,"Hong Kong, Hong Kong",hard,2025-01-05,https://www.atptour.com/en/tournaments/hong-ko...
2,8998,Adelaide International,ATP 250,"Adelaide, Australia",hard,2025-01-11,https://www.atptour.com/en/tournaments/adelaid...
3,301,ASB Classic,ATP 250,"Auckland, New Zealand",hard,2025-01-11,https://www.atptour.com/en/tournaments/aucklan...
4,580,Australian Open,Grand Slam,"Melbourne, Australia",hard,2025-01-26,https://www.atptour.com/en/tournaments/austral...
...,...,...,...,...,...,...,...
56,352,ATP Masters 1000 Paris,ATP 1000,"Paris, France",hard (indoor),2024-11-03,https://www.atptour.com/en/tournaments/paris/3...
57,4787,Belgrade,ATP 250,"Belgrade, Serbia",clay,2024-11-09,https://www.atptour.com/en/tournaments/belgrad...
58,341,Metz,ATP 250,"Metz, France",hard (indoor),2024-11-09,https://www.atptour.com/en/tournaments/metz/34...
59,605,Nitto ATP Finals,Nitto ATP Finals,"Turin, Italy",hard (indoor),2024-11-17,https://www.atptour.com/en/tournaments/nitto-a...


In [13]:
df_all_tournament = insert_surface(df_all_tournament)

ValueError: cannot insert surface, already exists

Save tournaments with surface

In [14]:
df_all_tournament.to_csv('/Users/samueleferrucci/Documents/Coding/Projects/Tennis ML/data/clean/all_tournaments.csv', sep=',', columns=df_all_tournament.columns, index=False)
df_all_tournament.to_csv('s3://matchedge-pipeline/data/clean/all_tournaments.csv', sep=',', columns=df_all_tournament.columns, index=False)

---
## Combine Relevant Data Into One Table

In [ ]:
df_all_tournament, df_all_stats_trial, df_results_trial, df_top

---
## Join Newly Scraped Data to All Existing Tables